In [1]:
#Listing 6.5
# import Session from the snowflake.snowpark package
from snowflake.snowpark import Session
# import data types from the snowflake.snowpark package
from snowflake.snowpark.types import StructType, StructField, DateType, BooleanType
# import json package for reading connection parameters
import json
# import date and timedelta from the datetime package for generating dates
from datetime import date, timedelta
# install the holidays package using pip or conda
# import the holidays package to determine whether a given date is a holiday
import holidays

In [2]:
#Listing 6.6
# define a function that returns True if p_date is a holiday in p_country
def is_holiday(p_date, p_country):
    # get a list of all holidays in p_country
    all_holidays = holidays.country_holidays(p_country)
    # return True if p_date is a holiday, otherwise return false
    if p_date in all_holidays:
        return True
    else:
        return False


In [3]:
#Listing 6.7
# generate a list of dates starting from start_date followed by as many days as defined in the no_days variable
# define the start date
start_dt = date(2023, 1, 1)
# define number of days
# use the value 5 to generate a sample dimension with 5 days
#no_days = 5 
# change the value to 731 to generate dates for 731 days (years 2023 and 2024)
no_days = 731
# store consecutive dates starting from the start date in a list
dates = [(start_dt + timedelta(days=i)).isoformat() for i in range(no_days)]

In [5]:
#Listing 6.8
# create a list of lists that combines the list of dates with the output of the is_holiday() function
holiday_flags = [[d, is_holiday(d, 'US')] for d in dates]

# print the holiday_flags list of lists locally to check that the data is as expected
print(holiday_flags)

[['2023-01-01', True], ['2023-01-02', True], ['2023-01-03', False], ['2023-01-04', False], ['2023-01-05', False], ['2023-01-06', False], ['2023-01-07', False], ['2023-01-08', False], ['2023-01-09', False], ['2023-01-10', False], ['2023-01-11', False], ['2023-01-12', False], ['2023-01-13', False], ['2023-01-14', False], ['2023-01-15', False], ['2023-01-16', True], ['2023-01-17', False], ['2023-01-18', False], ['2023-01-19', False], ['2023-01-20', False], ['2023-01-21', False], ['2023-01-22', False], ['2023-01-23', False], ['2023-01-24', False], ['2023-01-25', False], ['2023-01-26', False], ['2023-01-27', False], ['2023-01-28', False], ['2023-01-29', False], ['2023-01-30', False], ['2023-01-31', False], ['2023-02-01', False], ['2023-02-02', False], ['2023-02-03', False], ['2023-02-04', False], ['2023-02-05', False], ['2023-02-06', False], ['2023-02-07', False], ['2023-02-08', False], ['2023-02-09', False], ['2023-02-10', False], ['2023-02-11', False], ['2023-02-12', False], ['2023-02-13'

In [6]:
#Refer to Listing 6.4
# read the credentials from a file
credentials = json.load(open('connection_parameters.json'))
# create a dictionary with the connection parameters
connection_parameters_dict = {
    "account": credentials["account"],
    "user": credentials["user"],
    "password": credentials["password"],
    "role": credentials["role"],
    "warehouse": credentials["warehouse"],
    "database": credentials["database"],
    "schema": credentials["schema"]  # optional
}  

# create a session object for the Snowpark session
my_session = Session.builder.configs(connection_parameters_dict).create()



In [7]:
#Listing 6.9
# create a data frame from the holiday_flags list of lists and define the schema as two columns:
# - column named "day" with data type DateType
# - column named "holiday_flg" with data type BooleanType
df = my_session.create_dataframe(
    holiday_flags, 
    schema = StructType(
        [StructField("day", DateType()), 
         StructField("holiday_flg", BooleanType())])
    )

In [10]:
# print the data frame to verify that it contains the correct data
df.collect()


[Row(DAY=datetime.date(2023, 1, 1), HOLIDAY_FLG=True),
 Row(DAY=datetime.date(2024, 12, 31), HOLIDAY_FLG=False),
 Row(DAY=datetime.date(2024, 12, 30), HOLIDAY_FLG=False),
 Row(DAY=datetime.date(2024, 12, 29), HOLIDAY_FLG=False),
 Row(DAY=datetime.date(2024, 12, 28), HOLIDAY_FLG=False),
 Row(DAY=datetime.date(2024, 12, 27), HOLIDAY_FLG=False),
 Row(DAY=datetime.date(2024, 12, 26), HOLIDAY_FLG=False),
 Row(DAY=datetime.date(2024, 12, 25), HOLIDAY_FLG=True),
 Row(DAY=datetime.date(2024, 12, 24), HOLIDAY_FLG=False),
 Row(DAY=datetime.date(2024, 12, 23), HOLIDAY_FLG=False),
 Row(DAY=datetime.date(2024, 12, 22), HOLIDAY_FLG=False),
 Row(DAY=datetime.date(2024, 12, 21), HOLIDAY_FLG=False),
 Row(DAY=datetime.date(2024, 12, 20), HOLIDAY_FLG=False),
 Row(DAY=datetime.date(2024, 12, 19), HOLIDAY_FLG=False),
 Row(DAY=datetime.date(2024, 12, 18), HOLIDAY_FLG=False),
 Row(DAY=datetime.date(2024, 12, 17), HOLIDAY_FLG=False),
 Row(DAY=datetime.date(2024, 12, 16), HOLIDAY_FLG=False),
 Row(DAY=datetime.

In [11]:

#Listing 6.10
# save the data frame to a Snowflake table named DIM_DATE and overwrite the table if it already exists
df.write.mode("overwrite").save_as_table("DIM_DATE")


In [ ]:
# close the Snowpark session
my_session.close()